# Volatility Dynamics 16 — Calibrating AM-T variance scale and density shape by horizon

**Experimental status: Historical exploratory; not prospective confirmation.** Frozen methodology: [VD 033](../../discussions/Volatility_Dynamics/033_chatgpt_vd16_horizon_calibration_preregistration.md).

This is **not** fitting Student-t to volatility: native five-minute returns still follow the *unchanged* VD15 Student-t transition process. A horizon-specific **forecast-density overlay** rescales predicted variance, adjusts mixture tail shape at constant mean variance, or does both. As a result, the calibrated forecasts are **not guaranteed to be a single horizon-consistent Markov process**.

Five-minute Gaussian/Student-t Markov innovations are unchanged; no future targets are used to produce a historical forecast. All test labels were already examined by earlier modules, so results remain exploratory.

Seven horizons: **5m, 10m, 15m, 30m, 60m, 120m, 240m**. Primary: 60m. Candidates: `AM_T`, `AM_T_SCALE`, `AM_T_SHAPE`, `AM_T_BOTH`, `AM_G` and `M2_student_t`.


## 0. Repository, environment and immutable protocol

In [1]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

ROOT=Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT/'src'/'data_engineering').exists():
    ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').exists(), 'Run the notebook within deribit-market-data repository'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as vd14
from data_engineering.modeling import student_t_return_markov as vd15
from data_engineering.modeling import markov_return_horizon_calibration as vd16

DATA=os.getenv('DERIBIT_DATA_ROOT')
assert DATA and Path(DATA).exists(), 'Set DERIBIT_DATA_ROOT in .env to the historic data root'
HIST=Path(DATA)/'history'/'price_5m'
assert HIST.is_dir(), f'Cannot find historic five-minute prices: {HIST}'
OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'16'
OUT.mkdir(exist_ok=True,parents=True)
ASSETS=('AVAX','BTC','ETH','HYPE','SOL','TRX','XRP')
PARTICLES=int(os.getenv('VD16_PARTICLES','256'))
BOOT=int(os.getenv('VD16_BOOTSTRAPS','1000'))
assert PARTICLES==vd16.PARTICLES and BOOT==vd16.BOOT, 'Binding spec uses 256 particles and 1000 bootstrap draws'
print(vd16.VERSION,'| HISTORICAL_EXPLORATORY | source',HIST)
print(json.dumps(vd16.registration(),indent=2,default=str))
START=time.monotonic()

VD16-0.1.0 | HISTORICAL_EXPLORATORY | source D:\MarketData\Deribit\history\price_5m
{
  "version": "VD16-0.1.0",
  "horizons": [
    5,
    10,
    15,
    30,
    60,
    120,
    240
  ],
  "primary_horizon_min": 60,
  "source": "VD15 native 5m Student-t RETURN transition and two-factor VAR state",
  "particle_count": 256,
  "max_calibration_origins_per_asset": 512,
  "max_selection_origins_per_asset": 512,
  "validation_calibration_fraction": 0.6,
  "min_horizon_embargo_min": 5,
  "log_scale_bounds": [
    -0.7,
    0.7
  ],
  "shape_bounds": [
    0.45,
    2.0
  ],
  "small_shrinkage_penalty": 0.002,
  "bootstrap_draws": 1000,
  "status": "HISTORICAL_EXPLORATORY",
  "confirmation": "Prior branches examined historical test period; requires fresh post-2026-10-14 data"
}


## 1. Scientific unit tests before loading real market data

In [2]:
import subprocess
p=subprocess.run([sys.executable,'-m','pytest','-q',
    str(ROOT/'tests'/'test_markov_return_horizon_calibration.py')],
    cwd=str(ROOT),text=True,capture_output=True)
print(p.stdout[-4000:],p.stderr[-1500:])
assert p.returncode==0, 'VD16 tests must pass before market-data evaluation' 

......                                                                   [100%]
6 passed in 10.38s
 


## 2. Causal five-minute data and seven exact future targets

The historical input is exclusively Binance spot close-to-close five-minute prices. Native 5m returns are censored across missing bars, target windows never cross TRAIN/VALID/TEST partitions, and the already-inspected sample ends strictly before 2026-09-19 UTC.

In [3]:
raw,excluded=vd14.load_returns(HIST,ASSETS)
panel=vd14.panelize(raw)
del raw
features=vd14.make_features(panel)
frame=vd14.make_targets(panel,features,vd16.HORIZONS)
print('Excluded assets:',excluded)
print(panel.groupby('split').agg(days=('date','nunique'),rows=('return_5m','size'),
    valid_return=('return_5m','count')).to_string())
print('Prepared in',round(time.monotonic()-START,1),'seconds')

Excluded assets: {'HYPE': 'insufficient history'}
       days    rows  valid_return
split                            
test    150  259200        259200
train   448  774138        774132
valid   150  259200        259200
Prepared in 23.8 seconds


## 3. Fit calibrated AM-T distributions and score TEST (historically inspected)

The original AM-T five-minute return kernel is trained on TRAIN and selects five-minute degrees of freedom using VALID. Horizon calibration parameters are learned on **early VALID**, with embargo before **late VALID**, which supplies model-family selection scores. Neither calibration stage uses TEST labels. Scoring remains exploratory because the same historical TEST period influenced prior research.

In [4]:
scores,boot,asset_scores,calibration_grid,params=vd16.run_all(
    frame,particles=PARTICLES,B=BOOT,cal_limit=vd16.CAL_ORIGINS_PER_ASSET)
print('Models per horizon:')
print(scores.groupby('horizon_min').agg(models=('model','nunique'),
    origins=('origins','first'),days=('test_days','first')).to_string())
assert len(scores)==len(vd16.HORIZONS)*len(vd16.MODEL_NAMES)
assert (scores.groupby('horizon_min').origins.nunique()==1).all()
print('Fitted and scored in',round(time.monotonic()-START,1),'seconds')

Models per horizon:
             models  origins  days
horizon_min                       
5                 6   259194   150
10                6   259188   150
15                6   259182   150
30                6   259164   150
60                6   259128   150
120               6   259056   150
240               6   258912   150
Fitted and scored in 642.5 seconds


## 4. Calibration settings, 7-horizon scores and paired bootstrap

In [5]:
display(calibration_grid.round(6))
display(scores.round(7))
display(boot[(boot.horizon_min==vd16.PRIMARY)&(boot.metric.isin(['nll','tail_logloss']))].round(7))
display(asset_scores.loc[asset_scores.horizon_min==vd16.PRIMARY].round(7))
print('WARNING: historical TEST was already inspected in VD13-15; these are descriptive results, not validation of a new edge.')

,horizon_min,family,log_scale,variance_multiplier,shape,fit_nll,select_nll,optimization_success,cal_origins,select_origins,select_boundary
0,5,AM_T,0.000000,1.000000,1.000000,-5.111329,-5.136173,True,3072,3072,2026-02-21 00:00:00+00:00
1,5,AM_T_SCALE,0.502514,1.652872,1.000000,-5.161295,-5.198290,True,3072,3072,2026-02-21 00:00:00+00:00
2,5,AM_T_SHAPE,0.000000,1.000000,1.050290,-5.111809,-5.136183,True,3072,3072,2026-02-21 00:00:00+00:00
3,5,AM_T_BOTH,0.599143,1.820559,1.246854,-5.168715,-5.204328,True,3072,3072,2026-02-21 00:00:00+00:00
4,10,AM_T,0.000000,1.000000,1.000000,-4.709206,-4.741323,True,3072,3072,2026-02-21 00:00:00+00:00
5,10,AM_T_SCALE,0.525257,1.690893,1.000000,-4.768030,-4.809452,True,3072,3072,2026-02-21 00:00:00+00:00
6,10,AM_T_SHAPE,0.000000,1.000000,1.255004,-4.718952,-4.749849,True,3072,3072,2026-02-21 00:00:00+00:00
7,10,AM_T_BOTH,0.629134,1.875985,1.461513,-4.785222,-4.823523,True,3072,3072,2026-02-21 00:00:00+00:00
8,15,AM_T,0.000000,1.000000,1.000000,-4.529275,-4.431329,True,3072,3072,2026-02-21 00:00:00+00:00
9,15,AM_T_SCALE,0.474955,1.607943,1.000000,-4.580616,-4.531807,True,3072,3072,2026-02-21 00:00:00+00:00


,horizon_min,model,origins,test_days,nll,tail_logloss,tail_brier,tail_predicted,tail_observed,interval_miss95
0,5,AM_T,259194,150,-5.354158,0.078921,0.016741,0.013444,0.018048,0.105006
1,5,AM_T_SCALE,259194,150,-5.390956,0.078054,0.017300,0.026421,0.018048,0.056749
2,5,AM_T_SHAPE,259194,150,-5.354957,0.078382,0.016733,0.013500,0.018048,0.105103
3,5,AM_T_BOTH,259194,150,-5.399202,0.077706,0.017244,0.028590,0.018048,0.050950
4,5,AM_G,259194,150,-5.340947,0.078895,0.016791,0.012169,0.018048,0.120265
5,5,M2_student_t,259194,150,-5.081550,0.081984,0.016794,0.009748,0.018048,0.231525
6,10,AM_T,259188,150,-5.017126,0.082234,0.016786,0.012939,0.018122,0.099704
7,10,AM_T_SCALE,259188,150,-5.043869,0.080253,0.017555,0.027162,0.018122,0.053023
8,10,AM_T_SHAPE,259188,150,-5.024589,0.079057,0.016747,0.013434,0.018122,0.099553
9,10,AM_T_BOTH,259188,150,-5.060350,0.079010,0.017418,0.030072,0.018122,0.046877


,horizon_min,comparison,metric,delta,q025,q975,test_days,origins
84,60,AM_T_SCALE - AM_T,nll,-0.019560,-0.027633,-0.011957,150,259128
85,60,AM_T_SCALE - AM_T,tail_logloss,-0.005644,-0.008511,-0.002857,150,259128
87,60,AM_T_SHAPE - AM_T,nll,-0.025707,-0.034907,-0.017995,150,259128
88,60,AM_T_SHAPE - AM_T,tail_logloss,-0.008097,-0.010650,-0.005895,150,259128
90,60,AM_T_BOTH - AM_T,nll,-0.049290,-0.063645,-0.037824,150,259128
91,60,AM_T_BOTH - AM_T,tail_logloss,-0.010219,-0.015072,-0.005893,150,259128
93,60,AM_T_BOTH - M2_student_t,nll,0.017908,0.013973,0.021387,150,259128
94,60,AM_T_BOTH - M2_student_t,tail_logloss,0.002842,0.001464,0.004310,150,259128
96,60,AM_T_BOTH - AM_T_SCALE,nll,-0.029730,-0.036635,-0.023947,150,259128
97,60,AM_T_BOTH - AM_T_SCALE,tail_logloss,-0.004575,-0.006713,-0.002651,150,259128


,horizon_min,asset,model,origins,nll,tail_logloss,tail_brier,tail_predicted,tail_observed,interval_miss95
144,60,AVAX,AM_T,43188,-3.684984,0.102006,0.017708,0.011203,0.018547,0.086922
145,60,AVAX,AM_T_SCALE,43188,-3.695701,0.095002,0.018201,0.021912,0.018547,0.054460
146,60,AVAX,AM_T_SHAPE,43188,-3.709885,0.091978,0.017656,0.012863,0.018547,0.084398
147,60,AVAX,AM_T_BOTH,43188,-3.726023,0.089163,0.018066,0.025674,0.018547,0.047976
148,60,AVAX,AM_G,43188,-3.602205,0.121371,0.017758,0.010307,0.018547,0.088705
149,60,AVAX,M2_student_t,43188,-3.741882,0.085510,0.017665,0.015438,0.018547,0.059646
150,60,BTC,AM_T,43188,-4.284745,0.132675,0.029088,0.024247,0.031467,0.083287
151,60,BTC,AM_T_SCALE,43188,-4.293829,0.128774,0.030133,0.043032,0.031467,0.051727
152,60,BTC,AM_T_SHAPE,43188,-4.311249,0.125671,0.028924,0.025403,0.031467,0.081180
153,60,BTC,AM_T_BOTH,43188,-4.324238,0.125297,0.029693,0.045472,0.031467,0.046819


## 5. Check numerical quadrature stability using VALID only

The 256-particle mixture approximation can make apparent small density gains. Compare 256 versus nested 512 particles at 5m, 60m and 240m using **late VALID only**. Do not use historical TEST results to alter quadrature settings.

In [6]:
particle_check=vd16.particle_sensitivity_validation(frame,
    params['native_return_kernels'],params['state_transitions'],params['calibrations'],
    horizons=(5,60,240),particles=PARTICLES)
display(particle_check.round(8))

,horizon_min,asset,family,origins,particles_low,particles_high,validation_nll_low,validation_nll_high,high_minus_low
0,5,AVAX,AM_T,96,256,512,-4.666072,-4.675546,-0.009475
1,5,AVAX,AM_T_BOTH,96,256,512,-4.805749,-4.802103,0.003646
2,5,BTC,AM_T,96,256,512,-5.210065,-5.210832,-0.000767
3,5,BTC,AM_T_BOTH,96,256,512,-5.245401,-5.245464,-0.000063
4,5,ETH,AM_T,96,256,512,-4.968773,-4.968948,-0.000175
5,5,ETH,AM_T_BOTH,96,256,512,-5.012195,-5.013001,-0.000805
6,5,SOL,AM_T,96,256,512,-4.937627,-4.935919,0.001708
7,5,SOL,AM_T_BOTH,96,256,512,-4.966377,-4.965850,0.000527
8,5,TRX,AM_T,96,256,512,-5.691251,-5.694309,-0.003058
9,5,TRX,AM_T_BOTH,96,256,512,-5.860904,-5.859424,0.001479


## 6. Historical-only research decision and compact saved outputs

In [7]:
decision=vd16.decision(scores,boot,params)
assert decision['status']=='HISTORICAL_EXPLORATORY'
assert decision['prospective_confirmation'] is False
print(json.dumps(decision,indent=2))

scores.to_csv(OUT/'16_scores.csv',index=False,float_format='%.10g')
boot.to_csv(OUT/'16_bootstrap.csv',index=False,float_format='%.10g')
asset_scores.to_csv(OUT/'16_per_asset.csv',index=False,float_format='%.10g')
calibration_grid.to_csv(OUT/'16_calibration_grid.csv',index=False,float_format='%.10g')
particle_check.to_csv(OUT/'16_particle_sensitivity.csv',index=False,float_format='%.10g')
(OUT/'16_decision.json').write_text(json.dumps(decision,indent=2),encoding='utf-8')
manifest={'module':'Volatility_Dynamics_16','version':vd16.VERSION,
          'registration':vd16.registration(),'history_directory':str(HIST),
          'sample_start':str(panel.state_time.min()),'sample_end':str(panel.state_time.max()),
          'assets':sorted(panel.underlying.unique().tolist()),'excluded_assets':excluded,
          'split_days':panel.groupby('split').date.nunique().to_dict(),
          'parameters_and_selection':params,
          'decision':decision,
          'scientific_caveat':'Already inspected historical test; no new confirmation or trading claim.'}
(OUT/'16_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved seven compact results files to:',OUT)
print('No origin-level predictions, raw market data or DuckDB modifications.')

{
  "status": "HISTORICAL_EXPLORATORY",
  "primary_horizon_min": 60,
  "n_test_days": 150,
  "flags": {
    "scale_beats_AM_T": true,
    "shape_beats_AM_T": true,
    "both_beats_AM_T": true,
    "both_beats_M2": false
  },
  "selected_using_validation_only": {
    "5": "AM_T_BOTH",
    "10": "AM_T_BOTH",
    "15": "AM_T_BOTH",
    "30": "AM_T_BOTH",
    "60": "AM_T_BOTH",
    "120": "AM_T_BOTH",
    "240": "AM_T_BOTH"
  },
  "prospective_confirmation": false,
  "trading_approved": false,
  "warning": "TEST segment previously viewed in VD13-15. No inferential novelty/confirmation."
}
Saved seven compact results files to: D:\Projects\deribit_market_data\notebooks\Volatility_Dynamics\16
No origin-level predictions, raw market data or DuckDB modifications.
